# Patch: Embed Item 1A and update multi-section checkpoints

Items 1 and 7 are already embedded in `_minilm_checkpoints_multi/`.
This notebook **only** processes `item_1A` (the correct column name),
patches it into the existing checkpoints, then recomputes drift for all sections + composite.

## 0. Mount Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

## 1. Configuration

In [ ]:
import os

CONFIG = {
    'filings_folder': '/content/drive/MyDrive/FML_project_4',
    'output_folder':  '/content/drive/MyDrive/FML_project_4',

    # The column name in the filing parquets for Item 1A
    'section_col':    'item_1A',

    # Key used in checkpoint parquets (lowercase for consistency)
    'emb_key':        'emb_item_1A',
    'nchunks_key':    'n_chunks_item_1A',

    # All three sections (for drift recomputation)
    'all_sections':   ['item_1', 'item_1A', 'item_7'],

    'model_name':     'sentence-transformers/all-MiniLM-L6-v2',
    'chunk_size':     400,
    'chunk_overlap':  50,
    'embed_batch':    64,
    'start_year':     2004,
    'end_year':       2025,
}

CHECKPOINT_DIR = os.path.join(CONFIG['output_folder'], '_minilm_checkpoints_multi')
print('Config OK')
print(f'Will embed column: {CONFIG["section_col"]}')
print(f'Will store as:     {CONFIG["emb_key"]}')

## 2. Dependencies

In [ ]:
import subprocess
subprocess.run(['pip', 'install', '-q',
    'sentence-transformers', 'transformers', 'torch', 'pyarrow', 'tqdm'], check=True)

import glob
import numpy as np
import pandas as pd
import torch
import gc
from tqdm.auto import tqdm
from sentence_transformers import SentenceTransformer
from transformers import AutoTokenizer
from sklearn.metrics.pairwise import cosine_similarity

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

## 3. Load model + tokenizer

In [ ]:
model     = SentenceTransformer(CONFIG['model_name'], device=DEVICE)
tokenizer = AutoTokenizer.from_pretrained(CONFIG['model_name'])
print(f'Model loaded  |  dim: {model.get_sentence_embedding_dimension()}')

## 4. Chunking + embedding helpers

In [ ]:
CHUNK_SIZE = CONFIG['chunk_size']
STRIDE     = CONFIG['chunk_overlap']

def chunk_text(text: str) -> list[str]:
    if not text or len(text.split()) < 10:
        return []
    enc = tokenizer(
        text,
        max_length=CHUNK_SIZE,
        stride=STRIDE,
        return_overflowing_tokens=True,
        truncation=True,
        padding=False,
        add_special_tokens=False,
    )
    return [
        tokenizer.decode(ids, skip_special_tokens=True)
        for ids in enc['input_ids']
        if len(ids) > 10
    ]

def embed_chunks(chunks: list[str]) -> np.ndarray | None:
    if not chunks:
        return None
    embs = model.encode(
        chunks,
        batch_size=CONFIG['embed_batch'],
        show_progress_bar=False,
        convert_to_numpy=True,
    )
    return embs.mean(axis=0)

print('Helpers OK')

## 5. Verify `item_1A` exists in filing parquets

Quick sanity check before running the full loop.

In [ ]:
pq_files = sorted(glob.glob(
    os.path.join(CONFIG['filings_folder'], '**', '*_filings.parquet'), recursive=True
))
print(f'Filing parquets found: {len(pq_files)}')

# Check first 5 files
sec_col = CONFIG['section_col']
for f in pq_files[:5]:
    df = pd.read_parquet(f, columns=None)
    has_col = sec_col in df.columns
    non_empty = 0
    if has_col:
        non_empty = df[sec_col].dropna().apply(lambda x: len(str(x).split()) > 10).sum()
    print(f'  {os.path.basename(f)}: cols={list(df.columns)}  |  '
          f'{sec_col} present={has_col}  |  non-empty rows={non_empty}')

## 6. Embed Item 1A and patch into existing checkpoints

For each CIK:
1. Load the existing checkpoint (has `emb_item_1`, `emb_item_7` already)
2. Load the filing parquet and embed `item_1A`
3. Add `emb_item_1A` column to the checkpoint
4. Recompute `emb_composite` as mean of all available section embeddings
5. Overwrite the checkpoint

CIKs that already have a non-null `emb_item_1A` are skipped.

In [ ]:
SEC_COL    = CONFIG['section_col']   # 'item_1A'
EMB_KEY    = CONFIG['emb_key']        # 'emb_item_1A'
NCHUNKS_KEY = CONFIG['nchunks_key']   # 'n_chunks_item_1A'
ALL_SECS   = CONFIG['all_sections']   # ['item_1', 'item_1A', 'item_7']

checkpoint_files = sorted(glob.glob(f'{CHECKPOINT_DIR}/*.parquet'))
print(f'Existing checkpoints: {len(checkpoint_files)}')

# Build a lookup: cik -> filing parquet path
filing_lookup = {}
for pq_path in pq_files:
    cik = os.path.basename(pq_path).replace('_filings.parquet', '').lstrip('0')
    if cik:
        filing_lookup[cik] = pq_path

patched  = 0
skipped  = 0
no_file  = 0
failed   = 0

for cf in tqdm(checkpoint_files, desc='Patching Item 1A'):
    cik = os.path.basename(cf).replace('.parquet', '')

    # ── Load existing checkpoint ──────────────────────────────────
    try:
        ckpt = pd.read_parquet(cf)
    except Exception:
        failed += 1
        continue

    # Skip if already patched (emb_item_1A exists and has non-null values)
    if EMB_KEY in ckpt.columns and ckpt[EMB_KEY].notna().any():
        skipped += 1
        continue

    # ── Find the corresponding filing parquet ─────────────────────
    if cik not in filing_lookup:
        no_file += 1
        continue

    try:
        filing_df = pd.read_parquet(filing_lookup[cik])
    except Exception:
        failed += 1
        continue

    if SEC_COL not in filing_df.columns:
        # Column doesn't exist in this file — store None
        ckpt[EMB_KEY]     = None
        ckpt[NCHUNKS_KEY] = 0
        ckpt.to_parquet(cf, index=False, compression='snappy')
        patched += 1
        continue

    # Prep filing data
    filing_df['cik']  = filing_df['cik'].astype(str).str.strip().str.lstrip('0')
    filing_df['year'] = pd.to_numeric(filing_df['year'], errors='coerce').astype('Int64')
    filing_df = filing_df[filing_df['year'].between(CONFIG['start_year'], CONFIG['end_year'])]
    filing_df = (filing_df.sort_values('filing_date', na_position='first')
                         .drop_duplicates(subset=['cik', 'year'], keep='last')
                         .set_index('year'))

    # ── Embed Item 1A for each year in the checkpoint ─────────────
    emb_list    = []
    nchunk_list = []
    for _, row in ckpt.iterrows():
        yr = int(row['year'])
        if yr in filing_df.index:
            text   = str(filing_df.loc[yr, SEC_COL] if pd.notna(filing_df.loc[yr].get(SEC_COL)) else '')
            # Handle case where .loc returns multiple rows
            if isinstance(text, pd.Series):
                text = str(text.iloc[-1])
        else:
            text = ''

        chunks = chunk_text(text)
        emb    = embed_chunks(chunks)
        emb_list.append(emb.tolist() if emb is not None else None)
        nchunk_list.append(len(chunks))

    ckpt[EMB_KEY]     = emb_list
    ckpt[NCHUNKS_KEY] = nchunk_list

    # ── Recompute composite embedding ─────────────────────────────
    new_composites = []
    for _, row in ckpt.iterrows():
        sec_embs = []
        for sec in ALL_SECS:
            ekey = f'emb_{sec}'
            if ekey in ckpt.columns and row.get(ekey) is not None:
                sec_embs.append(np.array(row[ekey], dtype=np.float32))
        if sec_embs:
            new_composites.append(np.mean(sec_embs, axis=0).tolist())
        else:
            new_composites.append(None)
    ckpt['emb_composite'] = new_composites

    # ── Overwrite checkpoint ──────────────────────────────────────
    ckpt.to_parquet(cf, index=False, compression='snappy')
    patched += 1

    if patched % 25 == 0:
        print(f'  ... {patched} CIKs patched')
        gc.collect()

print(f'\nDone.  Patched: {patched}  |  Skipped (already done): {skipped}  '
      f'|  No filing file: {no_file}  |  Failed: {failed}')

## 7. Recompute semantic drift from patched checkpoints

Now all checkpoints have `emb_item_1`, `emb_item_1A`, `emb_item_7`, and `emb_composite`.
Recompute drift for all four.

In [ ]:
DRIFT_PATH = os.path.join(CONFIG['output_folder'], 'semantic_drift_minilm_multi.parquet')

checkpoint_files = sorted(glob.glob(f'{CHECKPOINT_DIR}/*.parquet'))
print(f'Checkpoint files: {len(checkpoint_files)}')

ALL_SECS = CONFIG['all_sections']  # ['item_1', 'item_1A', 'item_7']
EMB_KEYS   = [f'emb_{s}' for s in ALL_SECS] + ['emb_composite']
DRIFT_KEYS = [f'drift_{s}' for s in ALL_SECS] + ['drift_composite']
SIM_KEYS   = [f'sim_{s}' for s in ALL_SECS] + ['sim_composite']

print(f'Computing drift for: {DRIFT_KEYS}')

drift_rows: list[dict] = []

for cf in tqdm(checkpoint_files, desc='Computing drift'):
    try:
        cik_df = pd.read_parquet(cf).sort_values('year').reset_index(drop=True)
    except Exception:
        continue

    if len(cik_df) < 2:
        continue

    cik = cik_df['cik'].iloc[0]

    for i in range(1, len(cik_df)):
        record = {
            'cik':       cik,
            'year':      int(cik_df.loc[i, 'year']),
            'year_prev': int(cik_df.loc[i - 1, 'year']),
        }

        for emb_key, drift_key, sim_key in zip(EMB_KEYS, DRIFT_KEYS, SIM_KEYS):
            emb_curr = cik_df.loc[i, emb_key] if emb_key in cik_df.columns else None
            emb_prev = cik_df.loc[i - 1, emb_key] if emb_key in cik_df.columns else None

            if emb_curr is None or emb_prev is None:
                record[drift_key] = np.nan
                record[sim_key]   = np.nan
            else:
                v_curr = np.array(emb_curr, dtype=np.float32).reshape(1, -1)
                v_prev = np.array(emb_prev, dtype=np.float32).reshape(1, -1)
                sim    = float(cosine_similarity(v_curr, v_prev)[0, 0])
                record[drift_key] = round(1.0 - sim, 6)
                record[sim_key]   = round(sim, 6)

        drift_rows.append(record)

drift_df = pd.DataFrame(drift_rows)
drift_df['year'] = drift_df['year'].astype('Int64')

# Backward compat
drift_df['semantic_drift'] = drift_df['drift_item_1']
drift_df['cosine_sim']     = drift_df['sim_item_1']

drift_df.to_parquet(DRIFT_PATH, index=False, compression='snappy')
print(f'\nSaved: {DRIFT_PATH}')
print(f'  {len(drift_df):,} rows, {drift_df["cik"].nunique()} CIKs')
print(f'  Columns: {list(drift_df.columns)}')

## 8. Summary statistics — all sections (including patched Item 1A)

In [ ]:
SECTION_LABELS = {
    'drift_item_1':     'Item 1 (Business)',
    'drift_item_1A':    'Item 1A (Risk Factors)',
    'drift_item_7':     'Item 7 (MD&A)',
    'drift_composite':  'Composite (mean of 3)',
}

print('SEMANTIC DRIFT — all-MiniLM-L6-v2 — Multi-Section Summary')
print('=' * 70)
print(f'Total CIKs:  {drift_df["cik"].nunique()}')
print(f'Total rows:  {len(drift_df):,}')
print(f'Years:       {drift_df["year"].min()} – {drift_df["year"].max()}')
print()

for col, label in SECTION_LABELS.items():
    if col not in drift_df.columns:
        print(f'--- {label} ---  COLUMN MISSING')
        continue
    valid = drift_df[col].dropna()
    print(f'--- {label} ---')
    print(f'  Valid obs: {len(valid):,}  |  Missing: {drift_df[col].isna().sum():,}')
    if len(valid) > 0:
        print(f'  Mean:   {valid.mean():.4f}  |  Median: {valid.median():.4f}  |  Std: {valid.std():.4f}')
        print(f'  Min:    {valid.min():.4f}  |  Max:    {valid.max():.4f}')
    print()

# Correlation matrix
avail_cols = [c for c in SECTION_LABELS.keys() if c in drift_df.columns]
print('\nCorrelation matrix between drift signals:')
corr = drift_df[avail_cols].corr()
corr.index   = [SECTION_LABELS[c] for c in corr.index]
corr.columns = [SECTION_LABELS[c] for c in corr.columns]
print(corr.round(3).to_string())

## 9. Export individual section drift files

In [ ]:
OUTPUT_DIR = CONFIG['output_folder']

EXPORT_MAP = {
    'drift_item_1':    'semantic_drift_minilm_item1.parquet',
    'drift_item_1A':   'semantic_drift_minilm_item1a.parquet',
    'drift_item_7':    'semantic_drift_minilm_item7.parquet',
    'drift_composite': 'semantic_drift_minilm_composite.parquet',
}

base_cols = ['cik', 'year', 'year_prev']

for drift_col, filename in EXPORT_MAP.items():
    sim_col = drift_col.replace('drift_', 'sim_')
    if drift_col not in drift_df.columns:
        print(f'SKIP: {drift_col} not in dataframe')
        continue
    cols = base_cols + [drift_col]
    if sim_col in drift_df.columns:
        cols.append(sim_col)
    out = drift_df[cols].copy()
    out = out.rename(columns={drift_col: 'semantic_drift'})
    if sim_col in out.columns:
        out = out.rename(columns={sim_col: 'cosine_sim'})
    path = os.path.join(OUTPUT_DIR, filename)
    out.to_parquet(path, index=False, compression='snappy')
    valid_n = out['semantic_drift'].notna().sum()
    print(f'Saved: {filename}  ({valid_n:,} valid obs)')

# Backward-compatible
compat = drift_df[base_cols + ['semantic_drift', 'cosine_sim']].copy()
compat.to_parquet(os.path.join(OUTPUT_DIR, 'semantic_drift_minilm.parquet'),
                  index=False, compression='snappy')
print(f'\nBackward-compatible file saved: semantic_drift_minilm.parquet')
print('All exports complete.')

## 10. Diagnostic plots

In [ ]:
import matplotlib.pyplot as plt

SECTION_LABELS = {
    'drift_item_1':     'Item 1 (Business)',
    'drift_item_1A':    'Item 1A (Risk Factors)',
    'drift_item_7':     'Item 7 (MD&A)',
    'drift_composite':  'Composite (mean of 3)',
}
COLORS = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

# Only plot sections that have data
plot_items = [(col, label, color)
              for (col, label), color in zip(SECTION_LABELS.items(), COLORS)
              if col in drift_df.columns and drift_df[col].notna().sum() > 0]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

ax = axes[0]
for col, label, color in plot_items:
    annual = drift_df.dropna(subset=[col]).groupby('year')[col].mean()
    ax.plot(annual.index, annual.values, marker='o', markersize=4,
            color=color, linewidth=1.5, label=label)
ax.set_title('Mean Semantic Drift per Year by Section', fontsize=11)
ax.set_xlabel('Year'); ax.set_ylabel('1 − cosine similarity')
ax.legend(fontsize=8); ax.grid(linestyle='--', alpha=0.4)

ax = axes[1]
for col, label, color in plot_items:
    vals = drift_df[col].dropna()
    ax.hist(vals, bins=50, color=color, alpha=0.4, label=label, density=True)
ax.set_title('Pooled Distribution of Semantic Drift', fontsize=11)
ax.set_xlabel('Semantic Drift'); ax.set_ylabel('Density')
ax.legend(fontsize=8); ax.grid(axis='y', linestyle='--', alpha=0.4)

plt.suptitle('Semantic Drift — Multi-Section Comparison (with Item 1A patched)',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(CONFIG['output_folder'], 'semantic_drift_multi_section_summary.png'),
            dpi=150, bbox_inches='tight')
plt.show()
print('Done.')